# PFE ML — Phase F: Per-Segment Performance Of The Tuned HGB Winner

Phase D showed that legal form, sector (NAF code), and company age carry most of the model's predictive weight. That raises a natural follow-up question: **does the model work as well for every segment, or is its performance concentrated in a few dominant groups?**

If AP and AUC are similar across activity sections, legal forms, and company-age buckets, the model is *robust* — appropriate for a general-purpose SME risk scorer. If one or two segments dominate and the rest perform near-random, the model is *over-fitted to a subgroup* and the headline number is misleading.

## Methodology

- **Model**: HGB with Phase B tuned hyperparameters, refitted on years 2017-2022 with 2023 held out (same as Phase C and Phase D canonical model).
- **Segmentation axes**: 
  - **NAF section** — the first letter of `activity_code` (A=agriculture, C=industry, F=construction, G=trade, I=hospitality, etc.). Captures the broad sector.
  - **Legal form bucket** — the leading two digits of `legal_category_code` group similar forms (e.g., `54xx` = SARL, `56xx` = SAS, `10xx` = entreprise individuelle, `92xx` = association).
  - **Company age bucket** — `<3y`, `3-10y`, `10-20y`, `>20y`. Survival-curve intuition predicts the youngest bucket is hardest.
- **Metrics**: AP, AUC, base rate (positive %), and segment size, computed per cell on the 2023 held-out set.
- **Minimum cell size**: 1,000 rows. Segments smaller than that get reported in an aggregate `OTHER` bucket so per-cell AP estimates remain meaningful.

## What this notebook produces

Under `ml-artifacts/segments_phase_f/`:
- `segment_performance_naf.csv`, `_legal_form.csv`, `_age.csv` — per-bucket AP / AUC / size.
- One plot per axis showing AP and base-rate (so we can tell whether a high AP is real lift or just a high-base-rate floor).
- `segment_summary.md` — one paragraph per axis you can paste into the thesis.

## 1. Runtime And Constants

**CPU runtime.** Same fit cost as Phase D (~8 min). Per-segment scoring is fast (<1 min). Total ~10-12 min.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/zribi1/pfein.git'
BRANCH = 'data-extraction'
REPO_DIR = '/content/pfein'
BACKEND_DIR = f'{REPO_DIR}/back_end'

DRIVE_ROOT = '/content/drive/MyDrive/PFE ML Data/pfe_data'
DATA_LAKE = f'{DRIVE_ROOT}/data-lake'
ARTIFACTS_DIR = f'{DRIVE_ROOT}/ml-artifacts'
DUCKDB_TMP = '/content/pfein_duckdb_tmp'

TARGET = 'continuity_risk_12m_label'
TRAIN_MAX_ROWS = 2_000_000
START_YEAR = 2017
TEST_YEAR = 2023                # Phase C canonical year
MIN_CELL_SIZE = 1_000           # smaller segments fold into OTHER

FEATURES_GLOB = f'{DATA_LAKE}/features/company_year_features/**/*.parquet'
LABELS_GLOB = f'{DATA_LAKE}/features/risk_labels/**/*.parquet'

TUNED_HGB_PARAMS = Path(ARTIFACTS_DIR) / 'tuned_params_hgb.json'
PHASE_F_DIR = Path(ARTIFACTS_DIR) / 'segments_phase_f'
PHASE_F_DIR.mkdir(parents=True, exist_ok=True)

Path(DUCKDB_TMP).mkdir(parents=True, exist_ok=True)

print('TEST_YEAR      =', TEST_YEAR)
print('MIN_CELL_SIZE  =', MIN_CELL_SIZE)
print('OUT_DIR        =', PHASE_F_DIR)

## 2. Pull Code And Install Dependencies

In [ ]:
import os

if not Path(REPO_DIR).exists():
    !git clone --branch "$BRANCH" "$REPO_URL" "$REPO_DIR"

%cd $REPO_DIR
!git fetch origin
!git switch "$BRANCH" || git switch -c "$BRANCH" "origin/$BRANCH"
!git pull --ff-only origin "$BRANCH"
%cd $BACKEND_DIR

os.environ['DUCKDB_TEMP_DIRECTORY'] = DUCKDB_TMP
!pip install -q -r collabs/requirements-colab.txt

In [ ]:
import json

if not TUNED_HGB_PARAMS.exists():
    raise SystemExit(
        f'Phase B output missing: {TUNED_HGB_PARAMS}.\n'
        f'Run Phase B (collabs/pfe_ml_colab_tuning_phase_b.ipynb) first.'
    )
tuned_params = json.loads(TUNED_HGB_PARAMS.read_text(encoding='utf-8'))
print('HGB tuned params:')
for k, v in tuned_params.items():
    print(f'  {k}: {v}')

## 3. Load Data (2M Hash Sample, Years ≤ 2023)

In [ ]:
import math, duckdb, pandas as pd, numpy as np
from app.tools.train_continuity_model import EXCLUDE_COLUMNS

filters = [
    f'l."{TARGET}" IS NOT NULL',
    f'f.prediction_year >= {START_YEAR}',
    f'f.prediction_year <= {TEST_YEAR}',
]
where_sql = ' AND '.join(filters)

con = duckdb.connect()
total_rows = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql}
""").fetchone()[0]
print(f'Eligible rows (≤ {TEST_YEAR}): {total_rows:,}')

modulus = 1_000_000
threshold = max(1, min(modulus, math.ceil((TRAIN_MAX_ROWS / total_rows) * modulus * 1.15)))
row_hash = "hash(CAST(f.siren AS VARCHAR) || ':' || CAST(f.prediction_year AS VARCHAR))"
feature_cols = con.execute(
    f"DESCRIBE SELECT * FROM read_parquet('{FEATURES_GLOB}', union_by_name=true)"
).df()['column_name'].tolist()
select_cols = [c for c in feature_cols if c not in EXCLUDE_COLUMNS]
select_sql = ', '.join(f'f."{c}"' for c in select_cols)

df = con.execute(f"""
    SELECT {select_sql}, l."{TARGET}"
    FROM read_parquet('{FEATURES_GLOB}', union_by_name=true) f
    JOIN read_parquet('{LABELS_GLOB}', union_by_name=true) l USING (siren, prediction_year)
    WHERE {where_sql} AND {row_hash} % {modulus} < {threshold}
    ORDER BY {row_hash}
    LIMIT {TRAIN_MAX_ROWS}
""").df()
con.close()

print(f'Loaded shape: {df.shape}')
print(f'Years: {sorted(df["prediction_year"].unique())}')
print(f'Class balance: {df[TARGET].value_counts().to_dict()}')

In [ ]:
y = df[TARGET].astype(int)
feature_columns = [c for c in df.columns if c not in EXCLUDE_COLUMNS and c != TARGET]
X = df[feature_columns].copy()
for col in X.columns:
    if pd.api.types.is_bool_dtype(X[col]):
        X[col] = X[col].astype(float)

numeric_columns = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
categorical_columns = [c for c in X.columns if c not in numeric_columns]

train_mask = X['prediction_year'] < TEST_YEAR
X_train, X_test = X[train_mask].reset_index(drop=True), X[~train_mask].reset_index(drop=True)
y_train, y_test = y[train_mask].reset_index(drop=True), y[~train_mask].reset_index(drop=True)
print(f'Train (years < {TEST_YEAR}): {len(X_train):,} rows, {y_train.sum():,} positives')
print(f'Test  (year = {TEST_YEAR}):    {len(X_test):,} rows, {y_test.sum():,} positives')

## 4. Fit The 2023-Test HGB With Tuned Params And Score The Test Set

In [ ]:
import importlib, time, app.tools.train_continuity_model as _tcm
importlib.reload(_tcm)
from app.tools.train_continuity_model import _build_model_pipeline

train_pos = int((y_train == 1).sum())
train_neg = int((y_train == 0).sum())

pipeline = _build_model_pipeline(
    family='hgb',
    categorical_columns=categorical_columns,
    train_positive_count=train_pos,
    train_negative_count=train_neg,
    extra_params=tuned_params,
)

print('Fitting tuned HGB on years 2017–2022...')
start = time.time()
pipeline.fit(X_train, y_train)
print(f'Done in {(time.time()-start)/60:.1f} min')

from sklearn.metrics import average_precision_score, roc_auc_score
y_proba_test = pipeline.predict_proba(X_test)[:, 1]
ap = average_precision_score(y_test, y_proba_test)
auc = roc_auc_score(y_test, y_proba_test)
print(f'\nOverall 2023-test AP:  {ap:.4f}')
print(f'Overall 2023-test AUC: {auc:.4f}')

## 5. Build Segment Labels

Three orthogonal segmentation axes. Each row in `X_test` gets a single bucket per axis. NAF section uses the first letter of `activity_code`; legal-form bucket uses the first two digits of `legal_category_code`; age bucket comes from `company_age_years`.

In [ ]:
def naf_section(activity_code):
    if not isinstance(activity_code, str) or len(activity_code) < 1:
        return 'unknown'
    return activity_code[0].upper()

def legal_form_bucket(legal_category_code):
    if not isinstance(legal_category_code, str) or len(legal_category_code) < 2:
        if legal_category_code is None or (isinstance(legal_category_code, float) and np.isnan(legal_category_code)):
            return 'unknown'
        return str(legal_category_code)[:2] if legal_category_code else 'unknown'
    return legal_category_code[:2]

def age_bucket(age_years):
    if age_years is None or (isinstance(age_years, float) and np.isnan(age_years)):
        return 'unknown'
    if age_years < 3:
        return '<3y'
    if age_years < 10:
        return '3-10y'
    if age_years < 20:
        return '10-20y'
    return '>20y'

segments = pd.DataFrame({
    'naf_section': X_test['activity_code'].astype(object).map(naf_section),
    'legal_form_bucket': X_test['legal_category_code'].astype(object).map(legal_form_bucket),
    'age_bucket': X_test['company_age_years'].map(age_bucket),
    'y_true': y_test.values,
    'y_proba': y_proba_test,
})

print('Segment label preview:')
print(segments.head().to_string(index=False))
print(f'\nUnique NAF sections: {segments["naf_section"].nunique()}')
print(f'Unique legal-form buckets: {segments["legal_form_bucket"].nunique()}')
print(f'Age buckets: {sorted(segments["age_bucket"].unique())}')

## 6. Per-Segment Metrics

For each bucket: count, positive rate (base rate), AP, AUC. Buckets with fewer than `MIN_CELL_SIZE` rows are folded into an `OTHER` aggregate so each reported cell has enough signal for a stable AP/AUC. Buckets with zero positives can't have an AP/AUC and are flagged.

In [ ]:
def per_segment(seg_df, axis):
    counts = seg_df[axis].value_counts()
    big = counts[counts >= MIN_CELL_SIZE].index.tolist()
    work = seg_df.copy()
    work[axis] = work[axis].where(work[axis].isin(big), other='OTHER')
    rows = []
    for bucket, sub in work.groupby(axis, sort=False):
        n = len(sub)
        n_pos = int(sub['y_true'].sum())
        base_rate = n_pos / n if n else float('nan')
        if n_pos == 0 or n_pos == n:
            ap_b, auc_b = float('nan'), float('nan')
        else:
            ap_b = average_precision_score(sub['y_true'], sub['y_proba'])
            auc_b = roc_auc_score(sub['y_true'], sub['y_proba'])
        lift = ap_b / base_rate if (base_rate and not math.isnan(ap_b)) else float('nan')
        rows.append({
            'segment': bucket,
            'n_rows': n,
            'n_positives': n_pos,
            'base_rate': base_rate,
            'average_precision': ap_b,
            'roc_auc': auc_b,
            'ap_over_base_rate_lift': lift,
        })
    return pd.DataFrame(rows).sort_values('n_rows', ascending=False).reset_index(drop=True)

naf_perf = per_segment(segments, 'naf_section')
legal_perf = per_segment(segments, 'legal_form_bucket')
age_perf = per_segment(segments, 'age_bucket')

print('=== By NAF section ===')
print(naf_perf.to_string(index=False))
print('\n=== By legal-form bucket ===')
print(legal_perf.to_string(index=False))
print('\n=== By company age bucket ===')
print(age_perf.to_string(index=False))

naf_perf.to_csv(PHASE_F_DIR / 'segment_performance_naf.csv', index=False)
legal_perf.to_csv(PHASE_F_DIR / 'segment_performance_legal_form.csv', index=False)
age_perf.to_csv(PHASE_F_DIR / 'segment_performance_age.csv', index=False)
print(f'\nSaved per-segment CSVs to {PHASE_F_DIR}')

## 7. Plot AP And Base Rate Side By Side

Reporting AP alone is misleading because AP has a base-rate floor: a segment with 10% positives can't have AP below ~0.1 even from a random model. Plot both so you can see the *lift over base rate* — the actual signal contribution.

In [ ]:
import matplotlib.pyplot as plt

def plot_axis(perf_df, axis_name, out_path):
    perf_df = perf_df.dropna(subset=['average_precision']).copy()
    perf_df = perf_df.sort_values('n_rows', ascending=False)
    x = np.arange(len(perf_df))
    width = 0.35
    fig, ax = plt.subplots(figsize=(max(7, 0.7 * len(perf_df) + 4), 5))
    ax.bar(x - width/2, perf_df['average_precision'], width, label='AP', color='#0f766e')
    ax.bar(x + width/2, perf_df['base_rate'], width, label='base rate', color='#94a3b8')
    ax.set_xticks(x, perf_df['segment'].astype(str), rotation=30, ha='right')
    ax.set_ylabel('value')
    ax.set_title(f'Phase F — {axis_name}: AP vs base rate (sorted by segment size)')
    for xi, row in zip(x, perf_df.itertuples()):
        ax.text(xi - width/2, row.average_precision, f'{row.average_precision:.2f}',
                ha='center', va='bottom', fontsize=8)
        ax.text(xi + width/2, row.base_rate, f'{row.base_rate:.2f}',
                ha='center', va='bottom', fontsize=8)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.show()
    print(f'Saved: {out_path}')

plot_axis(naf_perf, 'NAF section', PHASE_F_DIR / 'segment_ap_naf.png')
plot_axis(legal_perf, 'legal-form bucket', PHASE_F_DIR / 'segment_ap_legal_form.png')
plot_axis(age_perf, 'company age bucket', PHASE_F_DIR / 'segment_ap_age.png')

## 8. Lift Ratio Per Segment (How Much Above Random?)

`ap_over_base_rate_lift = AP / base_rate`. A lift of 1 means the model is no better than predicting the base rate; lift ≥ 3-5 means real signal. Look for segments with low lift even if their absolute AP is high — those are the segments where the model is "riding the base rate" rather than ranking well.

In [ ]:
def plot_lift(perf_df, axis_name, out_path):
    perf_df = perf_df.dropna(subset=['ap_over_base_rate_lift']).copy()
    perf_df = perf_df.sort_values('ap_over_base_rate_lift', ascending=False)
    x = np.arange(len(perf_df))
    fig, ax = plt.subplots(figsize=(max(7, 0.6 * len(perf_df) + 4), 4.5))
    ax.bar(x, perf_df['ap_over_base_rate_lift'], color='#0f766e')
    ax.set_xticks(x, perf_df['segment'].astype(str), rotation=30, ha='right')
    ax.set_ylabel('AP / base rate (lift over random)')
    ax.axhline(1, color='#b91c1c', linestyle='--', label='no-signal baseline (lift = 1)')
    ax.set_title(f'Phase F — {axis_name}: lift over base rate')
    for xi, lift in zip(x, perf_df['ap_over_base_rate_lift']):
        ax.text(xi, lift, f'{lift:.1f}×', ha='center', va='bottom', fontsize=9)
    ax.grid(True, axis='y', alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(out_path, dpi=160)
    plt.show()
    print(f'Saved: {out_path}')

plot_lift(naf_perf, 'NAF section', PHASE_F_DIR / 'segment_lift_naf.png')
plot_lift(legal_perf, 'legal-form bucket', PHASE_F_DIR / 'segment_lift_legal_form.png')
plot_lift(age_perf, 'company age bucket', PHASE_F_DIR / 'segment_lift_age.png')

## 9. Verdict And Thesis Summary

In [ ]:
def summarize(perf_df, axis_label):
    valid = perf_df.dropna(subset=['average_precision', 'roc_auc'])
    ap_min, ap_max = valid['average_precision'].min(), valid['average_precision'].max()
    auc_min, auc_max = valid['roc_auc'].min(), valid['roc_auc'].max()
    lift_min, lift_max = valid['ap_over_base_rate_lift'].min(), valid['ap_over_base_rate_lift'].max()
    worst_ap_seg = valid.loc[valid['average_precision'].idxmin(), 'segment']
    best_ap_seg = valid.loc[valid['average_precision'].idxmax(), 'segment']
    worst_lift_seg = valid.loc[valid['ap_over_base_rate_lift'].idxmin(), 'segment']
    return (
        f'### {axis_label}\n'
        f'- Cells reported: {len(valid)} (after {MIN_CELL_SIZE}-row min-size cut)\n'
        f'- AP range: {ap_min:.3f} ({worst_ap_seg}) – {ap_max:.3f} ({best_ap_seg})\n'
        f'- AUC range: {auc_min:.3f} – {auc_max:.3f}\n'
        f'- Lift over base rate: {lift_min:.1f}× – {lift_max:.1f}× (worst: {worst_lift_seg})\n'
    )

summary_md = (
    '# Phase F — Per-Segment Performance Summary\n\n'
    f'Model: tuned HGB, trained 2017–2022, evaluated on {TEST_YEAR}.\n\n'
    + summarize(naf_perf, 'NAF section')
    + '\n' + summarize(legal_perf, 'Legal-form bucket')
    + '\n' + summarize(age_perf, 'Company age bucket')
    + '\n## Interpretation guide\n'
      '- A tight AP range (<2× spread) means the model is segment-robust.\n'
      '- A wide AUC range (>0.10 spread) means ranking quality varies materially across segments.\n'
      '- A segment with low lift (<1.5×) is one where the model adds little over predicting the base rate.\n'
)

summary_path = PHASE_F_DIR / 'segment_summary.md'
summary_path.write_text(summary_md, encoding='utf-8')
print(summary_md)
print(f'\nSaved: {summary_path}')

## 10. Decision Criteria For The Thesis

- **All segments lift ≥ 3× and AUC range < 0.05** → the model is broadly applicable. Report Phase B numbers as the overall result and cite Phase F to demonstrate segment-robustness.
- **One segment dominates (lift > 10×) while others lift < 2×** → the model is over-fitted to that segment. Two responses are acceptable: (a) report the overall number with a caveat and recommend retraining a per-segment ensemble, or (b) restrict the product's claimed scope to the segments where lift is real.
- **AUC range > 0.10 across segments** → ranking quality is unstable. Investigate whether the under-performing segments have meaningfully different feature distributions (e.g., financial-feature missingness rate, prevalence of legal events).

### Thesis paragraph

*"Phase F evaluated per-segment performance on the 2023 held-out set across three orthogonal axes — NAF section, legal-form bucket, and company-age bucket. AP varied from X.XX to Y.YY across the {N} reported NAF sections, AUC from A.AAA to B.BBB. Lift over the per-segment base rate ranged C.C× to D.D×, with the lowest lift observed in segment Z (interpret here why — small sample? low base rate? feature missingness?). The spread is consistent with a model that is broadly applicable across the French SME population rather than being dependent on a single dominant segment."*

Fill in the actual numbers from sections 6 and 9 when transferring to the thesis.

### What's next

After Phase F, the optional polish phases are **Phase E (calibration)** — reliability diagram + Brier score, important if the Angular frontend uses probability thresholds — and **Phase G (latency/cost)** — measure batch-prediction time and storage, important for the deployment chapter.